In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import joblib
import warnings
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# YAML konfigürasyon dosyasını yükle
# ---------------------------------------------------------------------------
with open("../configs/model_params.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

TEST_SIZE    = config["data"]["test_size"]         # 0.2
RANDOM_STATE = config["data"]["random_state"]      # 42
SCALER_TYPE  = config["preprocessing"]["numeric_scaler"]  # "StandardScaler"

print("✅ Konfigürasyon yüklendi:")
print(f"   Test oranı   : {TEST_SIZE}")
print(f"   Random state : {RANDOM_STATE}")
print(f"   Scaler       : {SCALER_TYPE}")

In [2]:
# ---------------------------------------------------------------------------
# Sütun grupları ve dizin yolları
# ---------------------------------------------------------------------------

# Orijinal sayısal sütunlar (02'den gelen)
NUMERICAL_COLS_ORIGINAL = [
    "Age", "Income", "LoanAmount", "CreditScore",
    "MonthsEmployed", "NumCreditLines", "InterestRate",
    "LoanTerm", "DTIRatio"
]

# Türetilecek finansal rasyolar
RATIO_COLS = [
    "LoanToIncome",
    "PaymentToIncome",
    "CreditAgePerLine",
    "TotalDebtBurden"
]

# Ölçeklenecek tüm sayısal sütunlar (orijinal + rasyolar)
NUMERICAL_COLS_ALL = NUMERICAL_COLS_ORIGINAL + RATIO_COLS  # 13 sütun

TARGET_COL = "Default"

INTERIM_DIR   = "../data/interim/"
PROCESSED_DIR = "../data/processed/"
MODELS_DIR    = "../models/"
FIGURES_DIR   = "../reports/figures/"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (14, 6), "figure.dpi": 120,
                     "axes.titlesize": 13, "axes.titleweight": "bold"})

print("✅ Sabitler hazır.")
print(f"   Orijinal sayısal sütun  : {len(NUMERICAL_COLS_ORIGINAL)}")
print(f"   Yeni rasyo sütunu       : {len(RATIO_COLS)}")
print(f"   Toplam sayısal sütun    : {len(NUMERICAL_COLS_ALL)}")

---
## 2. Temizlenmiş Veriyi Yükle

`02_preprocessing.ipynb`'nin ürettiği `loans_cleaned.csv` yüklenir.  
Bu dosya: ID drop + (isteğe bağlı IQR kırpma) + LabelEncoding yapılmış; split/scale edilmemiş tam veridir.

In [3]:
cleaned_path = f"{INTERIM_DIR}loans_cleaned.csv"

if not os.path.exists(cleaned_path):
    raise FileNotFoundError(
        f"{cleaned_path} bulunamadı. Lütfen önce '02_preprocessing.ipynb'i çalıştırın."
    )

df = pd.read_csv(cleaned_path)
print(f"✅ Temizlenmiş veri yüklendi: {df.shape[0]:,} satır × {df.shape[1]} sütun")
print(f"\n📌 Sütunlar: {list(df.columns)}")
df.head()

In [4]:
# Temel doğrulama kontrolleri
assert df.isnull().sum().sum() == 0, "❌ Eksik değer tespit edildi!"
assert df.select_dtypes(include='object').shape[1] == 0, "❌ String sütun tespit edildi!"
assert TARGET_COL in df.columns, f"❌ '{TARGET_COL}' sütunu bulunamadı!"

print("🔍 Veri Doğrulaması:")
print(f"   Eksik değer    : {df.isnull().sum().sum()} ✅")
print(f"   String sütun   : {df.select_dtypes(include='object').shape[1]} ✅")
print(f"\n⚖️  Sınıf dağılımı:")
vc = df[TARGET_COL].value_counts()
print(f"   Ödedi (0)    : {vc.get(0, 0):,} (%{vc.get(0, 0) / len(df) * 100:.1f})")
print(f"   Temerrüt (1) : {vc.get(1, 0):,} (%{vc.get(1, 0) / len(df) * 100:.1f})")
print(f"   Oran         : ~{vc.get(0, 0) / vc.get(1, 1):.1f}:1")

---
## 3. Feature Engineering — Finansal Rasyolar

Domain knowledge ile 4 yeni özellik türetilir. Bu rasyolar satır bazında hesaplandığından  
**global istatistik kullanılmaz** → split öncesi hesaplanmak güvenlidir (data leakage yok).

| Rasyo | Formül | Ekonomik Anlam |
|-------|--------|----------------|
| `LoanToIncome` | `LoanAmount / Income` | Yıllık gelire oranla borç büyüklüğü |
| `PaymentToIncome` | `aylık_ödeme / (Income/12)` | Aylık ödemenin aylık gelire oranı (front-end DTI) |
| `CreditAgePerLine` | `MonthsEmployed / (NumCreditLines + 1)` | Her kredi hattı başına düşen iş deneyimi |
| `TotalDebtBurden` | `(aylık_ödeme × LoanTerm) / Income` | Toplam geri ödeme tutarının yıllık gelire oranı |

**`PaymentToIncome` ve `TotalDebtBurden` hesabında aylık ödeme için annuity formülü:**  
`PMT = P × r / (1 − (1+r)^(−n))` &nbsp; → &nbsp; `r = InterestRate / 1200`, `n = LoanTerm (ay)`

> **Kapsam Dışı:** `Age × CreditScore` gibi anlamsız çarpım rasyoları veya hedef değişkenle  
> doğrudan ilişkili sütunlar (zaten modelin bildiği bilgiler) eklenmedi.

In [5]:
# ---------------------------------------------------------------------------
# Vektörize edilmiş annuity (eşit taksit) formülü
# PMT = P * r / (1 - (1+r)^(-n))  |  r=0 için özel durum: PMT = P/n
# ---------------------------------------------------------------------------
r = df["InterestRate"] / 1200          # %15.23 → 0.01269 (aylık oran)
n = df["LoanTerm"]                     # ay cinsinden vade
P = df["LoanAmount"]                   # kredi anaparası
monthly_income = df["Income"] / 12     # aylık gelir

monthly_pmt = np.where(
    r < 1e-10,
    P / np.maximum(n, 1),
    P * r / (1 - (1 + r) ** (-n))
)

# ---------------------------------------------------------------------------
# 4 finansal rasyo — tüm bölmeler için sıfır güvenliği
# ---------------------------------------------------------------------------
df["LoanToIncome"]     = P / np.maximum(df["Income"], 1)
df["PaymentToIncome"]  = monthly_pmt / np.maximum(monthly_income, 1)
df["CreditAgePerLine"] = df["MonthsEmployed"] / (df["NumCreditLines"] + 1)
df["TotalDebtBurden"]  = (monthly_pmt * n) / np.maximum(df["Income"], 1)

print("✅ 4 finansal rasyo hesaplandı:")
print(f"   Toplam sütun sayısı: {df.shape[1]}  ({df.shape[1] - 4} orijinal + 4 yeni)")
print(f"\n📊 Yeni özelliklerin özet istatistikleri:")
print(df[RATIO_COLS].describe().round(4).to_string())

In [6]:
# ---------------------------------------------------------------------------
# Yeni rasyoların hedef değişkenle korelasyonu
# Pozitif korelasyon = yüksek rasyo → daha fazla temerrüt riski (beklenen)
# ---------------------------------------------------------------------------
print("📈 Yeni rasyoların Default ile korelasyonu (Pearson):")
print()
for col in RATIO_COLS:
    corr = df[col].corr(df[TARGET_COL])
    direction = "(+) yüksek → daha fazla risk" if corr > 0 else "(-) yüksek → daha az risk"
    print(f"   {col:<22} : {corr:>+.4f}   {direction}")

In [7]:
# ---------------------------------------------------------------------------
# Yeni özellik dağılımları — Default = 0 vs 1 karşılaştırması
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

colors = {0: "#3498db", 1: "#e74c3c"}
labels = {0: "Ödedi (0)", 1: "Temerrüt (1)"}

for i, col in enumerate(RATIO_COLS):
    ax = axes[i]
    for cls in [0, 1]:
        vals = df.loc[df[TARGET_COL] == cls, col]
        # Aşırı uçları görselleştirmeden çıkar (99. persentil)
        cap = vals.quantile(0.99)
        sns.histplot(vals[vals <= cap], bins=60, kde=True, ax=ax,
                     color=colors[cls], alpha=0.55, label=labels[cls])
    ax.set_title(col)
    ax.set_xlabel("")
    ax.legend(fontsize=10)

fig.suptitle("Finansal Rasyolar — Default=0 vs Default=1",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}02b_ratio_distributions.png", bbox_inches="tight")
plt.show()
print("💾 Kaydedildi: 02b_ratio_distributions.png")

---
## 4. Feature / Target Ayrımı

Bağımsız değişkenler (**X**) ve hedef değişken (**y**) ayrılır.  
X, orijinal sütunlar + yeni 4 rasyo = **20 özellik** içerir.

In [8]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print(f"📊 Feature matrisi (X): {X.shape[0]:,} satır × {X.shape[1]} özellik")
print(f"🎯 Hedef vektörü  (y): {y.shape[0]:,} eleman")
print(f"\n📌 Feature sütunları ({X.shape[1]} adet):")
print(f"   Orijinal  : {NUMERICAL_COLS_ORIGINAL}")
print(f"   Yeni rasyo: {RATIO_COLS}")
print(f"   Kategorik : {[c for c in X.columns if c not in NUMERICAL_COLS_ALL]}")

---
## 5. Train / Test Split (Stratified)

Veriyi %80 eğitim / %20 test olarak ayırıyoruz.  
**Stratified split:** sınıf oranları (88.4% / 11.6%) her iki sette de korunur.  

```
✅ Doğru sıra : Rasyolar hesapla → Split → Scale
❌ Yanlış sıra: Scale → Split  (test verisinden bilgi sızar)
```

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y   # Sınıf oranlarını koru
)

print(f"✅ Stratified split tamamlandı (test_size={TEST_SIZE})")
print(f"\n{'Set':<8} {'Satır':>10} {'Ödedi (0)':>12} {'Temerrüt (1)':>14} {'Temerrüt (%)':>14}")
print("-" * 62)
for name, X_s, y_s in [("Train", X_train, y_train), ("Test", X_test, y_test)]:
    n0 = (y_s == 0).sum()
    n1 = (y_s == 1).sum()
    print(f"{name:<8} {len(y_s):>10,} {n0:>12,} {n1:>14,} {y_s.mean()*100:>13.1f}%")
print(f"\n📌 Feature sayısı: {X_train.shape[1]}")

---
## 6. Sayısal Değişkenleri Ölçeklendirme (StandardScaler)

Tüm 13 sayısal sütun (9 orijinal + 4 yeni rasyo) standartlaştırılır: μ=0, σ=1.

**⚠️ Kritik Kural — Data Leakage Önleme:**
```
✅ Doğru:  scaler.fit(X_train)  →  scaler.transform(X_test)
❌ Yanlış: scaler.fit(X_tümü)   →  sonra split et  (test'ten bilgi sızar)
❌ Yanlış: scaler.fit(X_test)   →  ölçek bozulur
```

**Not:** XGBoost ağaç tabanlı olduğu için ölçeklemeye duyarsızdır.  
Ancak pipeline tutarlılığı ve gelecekteki lineer modeller için standart uygulama budur.

In [10]:
scaler = StandardScaler()

# Fit SADECE train verisiyle — test verisi scaler'a gösterilmez
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[NUMERICAL_COLS_ALL] = scaler.fit_transform(X_train_scaled[NUMERICAL_COLS_ALL])

# Transform SADECE — fit yok
X_test_scaled[NUMERICAL_COLS_ALL] = scaler.transform(X_test_scaled[NUMERICAL_COLS_ALL])

print("✅ StandardScaler uygulandı (μ=0, σ=1)")
print("   → Scaler sadece train verisiyle fit edildi (data leakage yok)")
print(f"\n📊 Train istatistikleri (sayısal sütunlar — μ ve σ kontrol):")
stats = X_train_scaled[NUMERICAL_COLS_ALL].agg(["mean", "std"]).round(4)
print(stats.to_string())

In [11]:
# Ölçekleme sonrası dağılım — 4 yeni rasyo kontrol
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, col in zip(axes, RATIO_COLS):
    sns.histplot(X_train_scaled[col], bins=50, kde=True, ax=ax, color="#9b59b6", alpha=0.7)
    ax.axvline(0, color="#e74c3c", linestyle="--", linewidth=1.5, alpha=0.7)
    ax.set_title(f"{col}\nμ={X_train_scaled[col].mean():.3f}  σ={X_train_scaled[col].std():.3f}")
    ax.set_xlabel("")

fig.suptitle("Ölçekleme Sonrası Yeni Rasyo Dağılımları (Train)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}02b_scaled_ratios.png", bbox_inches="tight")
plt.show()
print("💾 Kaydedildi: 02b_scaled_ratios.png")

---
## 7. Çıktıları Kaydet

| Dosya | İçerik | Kullanım Yeri |
|-------|--------|---------------|
| `data/processed/X_train.csv` | Eğitim feature'ları (20 sütun) | Notebook 03 — Model eğitimi |
| `data/processed/X_test.csv` | Test feature'ları (20 sütun) | Notebook 03 — Değerlendirme |
| `data/processed/y_train.csv` | Eğitim hedef değişkeni | Notebook 03 |
| `data/processed/y_test.csv` | Test hedef değişkeni | Notebook 03 |
| `models/scaler.joblib` | StandardScaler nesnesi (13 sayısal sütun) | API — inference |

In [12]:
# ---------------------------------------------------------------------------
# İşlenmiş veriyi kaydet
# ---------------------------------------------------------------------------
X_train_scaled.to_csv(f"{PROCESSED_DIR}X_train.csv", index=False)
X_test_scaled.to_csv(f"{PROCESSED_DIR}X_test.csv",   index=False)
y_train.to_csv(f"{PROCESSED_DIR}y_train.csv", index=False)
y_test.to_csv(f"{PROCESSED_DIR}y_test.csv",   index=False)

print("✅ Split veriler kaydedildi:")
for fname in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    fpath = f"{PROCESSED_DIR}{fname}"
    print(f"   📄 {fname:<16} → {os.path.getsize(fpath) / 1024:>8.1f} KB")

# ---------------------------------------------------------------------------
# Scaler'ı kaydet (inference pipeline için)
# ---------------------------------------------------------------------------
scaler_path = f"{MODELS_DIR}scaler.joblib"
joblib.dump(scaler, scaler_path)

print(f"\n✅ Scaler kaydedildi:")
print(f"   📦 scaler.joblib → {MODELS_DIR}")
print(f"   Fit edilen sütunlar ({len(NUMERICAL_COLS_ALL)}): {NUMERICAL_COLS_ALL}")

In [13]:
# ---------------------------------------------------------------------------
# Doğrulama: Kaydedilen veriyi yeniden yükle
# ---------------------------------------------------------------------------
X_train_check = pd.read_csv(f"{PROCESSED_DIR}X_train.csv")
X_test_check  = pd.read_csv(f"{PROCESSED_DIR}X_test.csv")
y_train_check = pd.read_csv(f"{PROCESSED_DIR}y_train.csv").squeeze()
y_test_check  = pd.read_csv(f"{PROCESSED_DIR}y_test.csv").squeeze()
scaler_check  = joblib.load(scaler_path)

print("🔍 Doğrulama Sonuçları:")
print(f"   X_train boyut  : {X_train_check.shape} → {'✅' if X_train_check.shape == X_train.shape else '❌'}")
print(f"   X_test boyut   : {X_test_check.shape}  → {'✅' if X_test_check.shape == X_test.shape else '❌'}")
print(f"   y_train boyut  : {y_train_check.shape[0]:,} → {'✅' if len(y_train_check) == len(y_train) else '❌'}")
print(f"   y_test boyut   : {y_test_check.shape[0]:,} → {'✅' if len(y_test_check) == len(y_test) else '❌'}")
print(f"   Eksik değer    : {X_train_check.isnull().sum().sum() + X_test_check.isnull().sum().sum()} → {'✅' if X_train_check.isnull().sum().sum() == 0 else '❌'}")
print(f"   String sütun   : {X_train_check.select_dtypes('object').shape[1]} → {'✅' if X_train_check.select_dtypes('object').shape[1] == 0 else '❌'}")
print(f"   Scaler yükleme : {'✅' if scaler_check is not None else '❌'}")
print(f"   Scaler sütunlar: {len(scaler_check.feature_names_in_)} → {'✅' if len(scaler_check.feature_names_in_) == len(NUMERICAL_COLS_ALL) else '❌'}")
print(f"\n📌 X_train sütunları ({X_train_check.shape[1]}): {list(X_train_check.columns)}")

---
## 📋 Özet & Son Kontrol

| # | İşlem | Detay |
|---|-------|-------|
| 1 | Veri yükleme | `data/interim/loans_cleaned.csv` (02'nin çıktısı) |
| 2 | Finansal rasyolar | 4 yeni özellik → toplam 20 sütun |
| 3 | Feature/Target | X (20 sütun) + y (Default) |
| 4 | Train/Test split | 204.277 / 51.070 — stratified |
| 5 | Scaling | 13 sayısal sütun → StandardScaler (μ=0, σ=1) |
| 6 | Kaydetme | 4 CSV + `scaler.joblib` |

### API Inference Notu
Yeni bir müşteri verisi geldiğinde inference pipeline şu adımları izlemelidir:
1. `label_encoders.joblib` → kategorik sütunları encode et
2. 4 finansal rasyoyu hesapla (annuity formülü ile)
3. `scaler.joblib` → 13 sayısal sütunu transform et
4. Modele ver

### Sonraki Notebook (`03_xgboost_risk_model.ipynb`)
- `data/processed/X_train.csv` → 20 özellik ile yeniden eğitim
- SHAP açıklanabilirliği 4 yeni özelliği de kapsayacak

---
*Feature engineering tamamlandı. Veriler `data/processed/` klasörüne kaydedilmiştir.*